In [1]:
import pandas as pd

df=pd.read_json("D:/FM/5_EMA/artifacts/hist_data_5m.json")

df

,date,time,open,high,low,close,volume,symbol
0,2026-01-12,09:15:00,619.85,624.40,618.00,618.05,777449,VEDL
1,2026-01-12,09:20:00,618.00,618.55,615.60,615.70,316728,VEDL
2,2026-01-12,09:25:00,615.95,617.50,614.80,615.40,327267,VEDL
3,2026-01-12,09:30:00,615.30,618.00,615.30,617.45,369436,VEDL
4,2026-01-12,09:35:00,617.50,618.45,616.30,617.50,210330,VEDL
...,...,...,...,...,...,...,...,...
4867,2026-04-20,14:50:00,773.65,774.00,773.15,773.55,118496,VEDL
4868,2026-04-20,14:55:00,773.65,773.75,773.00,773.45,94050,VEDL
4869,2026-04-20,15:00:00,773.40,774.00,773.00,773.00,154418,VEDL
4870,2026-04-20,15:05:00,773.00,773.05,771.65,771.95,238109,VEDL


In [2]:
df['sma_5'] = df['close'].rolling(window=5).mean()

In [3]:
df['prev_sma_5']=df['sma_5'].shift(1)

In [4]:
df.head()

,date,time,open,high,low,close,volume,symbol,sma_5,prev_sma_5
0,2026-01-12,09:15:00,619.85,624.40,618.0,618.05,777449,VEDL,NaN,NaN
1,2026-01-12,09:20:00,618.00,618.55,615.6,615.70,316728,VEDL,NaN,NaN
2,2026-01-12,09:25:00,615.95,617.50,614.8,615.40,327267,VEDL,NaN,NaN
3,2026-01-12,09:30:00,615.30,618.00,615.3,617.45,369436,VEDL,NaN,NaN
4,2026-01-12,09:35:00,617.50,618.45,616.3,617.50,210330,VEDL,616.82,NaN


In [5]:
#df = df[['date','time','close','prev_sma_5']]
df = df[(df['time'] >= '09:20:00') & (df['time'] <= '15:00:00')]

In [6]:
df.head()

,date,time,open,high,low,close,volume,symbol,sma_5,prev_sma_5
1,2026-01-12,09:20:00,618.00,618.55,615.6,615.70,316728,VEDL,NaN,NaN
2,2026-01-12,09:25:00,615.95,617.50,614.8,615.40,327267,VEDL,NaN,NaN
3,2026-01-12,09:30:00,615.30,618.00,615.3,617.45,369436,VEDL,NaN,NaN
4,2026-01-12,09:35:00,617.50,618.45,616.3,617.50,210330,VEDL,616.82,NaN
5,2026-01-12,09:40:00,617.50,617.85,615.0,615.85,148872,VEDL,616.38,616.82


In [7]:
import pandas as pd
import numpy as np

# =========================
# 🧹 PREP
# =========================
df = df.copy()
df = df.sort_values(by=['date', 'time']).reset_index(drop=True)

df['date'] = pd.to_datetime(df['date'])
df['datetime'] = pd.to_datetime(df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['time'])

# =========================
# 🎯 ENTRY SIGNAL
# =========================
df['entry_signal'] = (
    (df['close'] > df['prev_sma_5']) &
    (df['close'].shift(1) <= df['prev_sma_5'].shift(1))
)

# =========================
# 💸 CHARGES FUNCTION (FYERS)
# =========================
def calculate_charges(entry_price, exit_price, quantity):
    
    buy_turnover = entry_price * quantity
    sell_turnover = exit_price * quantity
    turnover = buy_turnover + sell_turnover

    # Brokerage
    brokerage_buy = min(20, buy_turnover * 0.0003)
    brokerage_sell = min(20, sell_turnover * 0.0003)
    brokerage = brokerage_buy + brokerage_sell

    # STT
    stt = sell_turnover * 0.00025

    # Exchange
    txn = turnover * 0.0000345

    # SEBI
    sebi = turnover * 0.000001

    # Stamp duty
    stamp = buy_turnover * 0.00003

    # GST
    gst = 0.18 * (brokerage + txn)

    total_charges = brokerage + stt + txn + sebi + stamp + gst

    return total_charges

# =========================
# 🧠 TRADE ENGINE
# =========================
position = 0
entry_price = 0
stop_loss = 0
quantity = 1
cost = 0

initial_capital = 100000
capital = initial_capital

positions = []
trade_returns = []
pnl_list = []

for i in range(len(df)):
    
    row = df.loc[i]
    pnl = 0

    # ========= ENTRY =========
    if position == 0 and row['entry_signal'] and i > 0:
        entry_price = row['close']
        stop_loss = df.loc[i-1, 'low']
        
        quantity = int(capital // entry_price)
        
        if quantity > 0:
            cost = quantity * entry_price
            position = 1

    # ========= EXIT =========
    elif position == 1:
        
        exit_price = None

        # SL hit
        if row['low'] <= stop_loss:
            exit_price = stop_loss
        
        # SMA exit
        elif row['close'] < row['prev_sma_5']:
            exit_price = row['close']

        if exit_price is not None:
            exit_value = quantity * exit_price
            gross_pnl = exit_value - cost
            
            charges = calculate_charges(entry_price, exit_price, quantity)
            
            net_pnl = gross_pnl - charges
            
            capital += net_pnl
            pnl = net_pnl

            position = 0
            quantity = 0

    positions.append(position)
    trade_returns.append(pnl)
    pnl_list.append(pnl)

# =========================
# 💰 STORE RESULTS
# =========================
df['position'] = positions
df['pnl'] = pnl_list

# Equity curve
df['equity'] = initial_capital + df['pnl'].cumsum()

# =========================
# 📊 DAILY PnL
# =========================
daily_pnl = df.groupby(df['date'].dt.date)['pnl'].sum()

# =========================
# 📉 MAX DRAWDOWN
# =========================
rolling_max = df['equity'].cummax()
drawdown = df['equity'] / rolling_max - 1
max_drawdown = drawdown.min()

# =========================
# 📊 TRADE METRICS
# =========================
trades = df[df['pnl'] != 0].copy()
trades['trade_date'] = trades['date'].dt.date

total_trades = len(trades)
trades_per_day = trades.groupby('trade_date').size()

win_trades = trades[trades['pnl'] > 0]
loss_trades = trades[trades['pnl'] < 0]

win_rate = (len(win_trades) / total_trades * 100) if total_trades > 0 else 0

avg_win = win_trades['pnl'].mean() if len(win_trades) > 0 else 0
avg_loss = loss_trades['pnl'].mean() if len(loss_trades) > 0 else 0

gross_profit = win_trades['pnl'].sum()
gross_loss = abs(loss_trades['pnl'].sum())

profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.nan

loss_rate = 1 - (win_rate / 100)
expectancy = (win_rate/100 * avg_win) - (loss_rate * abs(avg_loss))

trade_efficiency = trades['pnl'].mean() if total_trades > 0 else 0

# =========================
# 📦 FINAL REPORT
# =========================
final_capital = df['equity'].iloc[-1]
total_profit = final_capital - initial_capital
return_pct = (total_profit / initial_capital) * 100

print("\n===== FINAL REPORT =====")
print(f"Initial Capital: ₹{initial_capital}")
print(f"Final Capital: ₹{round(final_capital,2)}")
print(f"Total Profit: ₹{round(total_profit,2)}")
print(f"Return: {round(return_pct,2)}%")

print("\n===== RISK =====")
print("Max Drawdown:", round(max_drawdown * 100, 2), "%")

print("\n===== TRADE METRICS =====")
print("Total Trades:", total_trades)
print("Win Rate:", round(win_rate, 2), "%")
print("Avg Win (₹):", round(avg_win, 2))
print("Avg Loss (₹):", round(avg_loss, 2))
print("Profit Factor:", round(profit_factor, 2))
print("Expectancy (₹):", round(expectancy, 2))
print("Trade Efficiency (₹):", round(trade_efficiency, 2))

print("\n===== TRADES PER DAY =====")
print(trades_per_day)

print("\n===== DAILY PnL =====")
print(daily_pnl)


===== FINAL REPORT =====
Initial Capital: ₹100000
Final Capital: ₹87189.97
Total Profit: ₹-12810.03
Return: -12.81%

===== RISK =====
Max Drawdown: -22.74 %

===== TRADE METRICS =====
Total Trades: 543
Win Rate: 23.39 %
Avg Win (₹): 688.79
Avg Loss (₹): -241.07
Profit Factor: 0.87
Expectancy (₹): -23.59
Trade Efficiency (₹): -23.59

===== TRADES PER DAY =====
trade_date
2026-01-12     6
2026-01-13     6
2026-01-14     8
2026-01-16     6
2026-01-19     8
              ..
2026-04-13     9
2026-04-15     6
2026-04-16    10
2026-04-17     6
2026-04-20    10
Length: 65, dtype: int64

===== DAILY PnL =====
date
2026-01-12     551.767362
2026-01-13    1572.197357
2026-01-14    4397.131758
2026-01-16     348.831295
2026-01-19   -1310.874837
                 ...     
2026-04-13     460.968809
2026-04-15    -410.247482
2026-04-16    1014.775808
2026-04-17     280.637328
2026-04-20   -1579.916063
Name: pnl, Length: 65, dtype: float64


After SL

In [8]:
import pandas as pd
import numpy as np

# =========================
# 🧹 PREP
# =========================
df = df.copy()
df = df.sort_values(by=['date', 'time']).reset_index(drop=True)

df['date'] = pd.to_datetime(df['date'])
df['datetime'] = pd.to_datetime(df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['time'])

# =========================
# 🎯 SIGNALS
# =========================
df['long_entry'] = (
    (df['close'] > df['prev_sma_5']) &
    (df['close'].shift(1) <= df['prev_sma_5'].shift(1))
)

df['short_entry'] = (
    (df['close'] < df['prev_sma_5']) &
    (df['close'].shift(1) >= df['prev_sma_5'].shift(1))
)

# =========================
# 💸 CHARGES (FYERS)
# =========================
def calculate_charges(entry_price, exit_price, quantity):
    buy_turnover = entry_price * quantity
    sell_turnover = exit_price * quantity
    turnover = buy_turnover + sell_turnover

    brokerage = min(20, buy_turnover * 0.0003) + min(20, sell_turnover * 0.0003)
    stt = sell_turnover * 0.00025
    txn = turnover * 0.0000345
    sebi = turnover * 0.000001
    stamp = buy_turnover * 0.00003
    gst = 0.18 * (brokerage + txn)

    return brokerage + stt + txn + sebi + stamp + gst

# =========================
# 🧠 TRADE ENGINE
# =========================
position = 0  # 1 = long, -1 = short
entry_price = 0
quantity = 1
cost = 0

initial_capital = 100000
capital = initial_capital

positions = []
pnl_list = []

for i in range(len(df)):
    
    row = df.loc[i]
    pnl = 0

    # ========= ENTRY =========
    if position == 0 and i > 0:
        
        if row['long_entry']:
            entry_price = row['close']
            quantity = int(capital // entry_price)

            if quantity > 0:
                cost = quantity * entry_price
                position = 1

        elif row['short_entry']:
            entry_price = row['close']
            quantity = int(capital // entry_price)

            if quantity > 0:
                cost = quantity * entry_price
                position = -1

    # ========= EXIT =========
    elif position != 0:
        
        prev_low = df.loc[i-1, 'low']
        prev_high = df.loc[i-1, 'high']
        exit_price = None

        # ===== LONG EXIT =====
        if position == 1:
            if row['close'] < prev_low or row['close'] < row['prev_sma_5']:
                exit_price = row['close']
                exit_value = quantity * exit_price
                gross_pnl = exit_value - cost

        # ===== SHORT EXIT =====
        elif position == -1:
            if row['close'] > prev_high or row['close'] > row['prev_sma_5']:
                exit_price = row['close']
                exit_value = quantity * exit_price
                gross_pnl = cost - exit_value  # reverse logic

        if exit_price is not None:
            charges = calculate_charges(entry_price, exit_price, quantity)
            net_pnl = gross_pnl - charges

            capital += net_pnl
            pnl = net_pnl

            position = 0
            quantity = 0

    positions.append(position)
    pnl_list.append(pnl)

# =========================
# 💰 RESULTS
# =========================
df['position'] = positions
df['pnl'] = pnl_list
df['equity'] = initial_capital + df['pnl'].cumsum()

# =========================
# 📊 METRICS
# =========================
daily_pnl = df.groupby(df['date'].dt.date)['pnl'].sum()

rolling_max = df['equity'].cummax()
drawdown = df['equity'] / rolling_max - 1
max_drawdown = drawdown.min()

trades = df[df['pnl'] != 0].copy()
trades['trade_date'] = trades['date'].dt.date

total_trades = len(trades)
trades_per_day = trades.groupby('trade_date').size()

win_trades = trades[trades['pnl'] > 0]
loss_trades = trades[trades['pnl'] < 0]

win_rate = (len(win_trades) / total_trades * 100) if total_trades > 0 else 0

avg_win = win_trades['pnl'].mean() if len(win_trades) > 0 else 0
avg_loss = loss_trades['pnl'].mean() if len(loss_trades) > 0 else 0

gross_profit = win_trades['pnl'].sum()
gross_loss = abs(loss_trades['pnl'].sum())

profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.nan

loss_rate = 1 - (win_rate / 100)
expectancy = (win_rate/100 * avg_win) - (loss_rate * abs(avg_loss))

# =========================
# 📦 REPORT
# =========================
final_capital = df['equity'].iloc[-1]

print("\n===== FINAL REPORT =====")
print(f"Initial Capital: ₹{initial_capital}")
print(f"Final Capital: ₹{round(final_capital,2)}")
print(f"Return: {round((final_capital/initial_capital - 1)*100,2)}%")

print("\nMax Drawdown:", round(max_drawdown * 100, 2), "%")

print("\n===== TRADE METRICS =====")
print("Total Trades:", total_trades)
print("Win Rate:", round(win_rate, 2), "%")
print("Profit Factor:", round(profit_factor, 2))
print("Expectancy (₹):", round(expectancy, 2))

print("\nTrades Per Day:")
print(trades_per_day)

print("\nDaily PnL:")
print(daily_pnl)


===== FINAL REPORT =====
Initial Capital: ₹100000
Final Capital: ₹47823.66
Return: -52.18%

Max Drawdown: -52.68 %

===== TRADE METRICS =====
Total Trades: 583
Win Rate: 20.07 %
Profit Factor: 0.49
Expectancy (₹): -89.5

Trades Per Day:
trade_date
2026-01-12     8
2026-01-13     6
2026-01-14     8
2026-01-16     6
2026-01-19     9
              ..
2026-04-13    10
2026-04-15     8
2026-04-16    10
2026-04-17     5
2026-04-20    11
Length: 65, dtype: int64

Daily PnL:
date
2026-01-12     498.191838
2026-01-13     -31.276421
2026-01-14   -1453.261314
2026-01-16      90.138906
2026-01-19   -1226.550248
                 ...     
2026-04-13    -657.015526
2026-04-15    -155.290336
2026-04-16    -927.967345
2026-04-17    -414.860358
2026-04-20    -416.642594
Name: pnl, Length: 65, dtype: float64


In [9]:
import pandas as pd
import numpy as np

# =========================
# 🧹 PREP
# =========================
df = df.copy()
df = df.sort_values(by=['date', 'time']).reset_index(drop=True)

df['date'] = pd.to_datetime(df['date'])
df['datetime'] = pd.to_datetime(df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['time'])

# =========================
# 🎯 SIGNALS
# =========================
df['long_entry'] = (
    (df['close'] > df['prev_sma_5']) &
    (df['close'].shift(1) <= df['prev_sma_5'].shift(1))
)

df['short_entry'] = (
    (df['close'] < df['prev_sma_5']) &
    (df['close'].shift(1) >= df['prev_sma_5'].shift(1))
)

# =========================
# 💸 CHARGES (FYERS)
# =========================
def calculate_charges(entry_price, exit_price, qty=1):
    buy_turnover = entry_price * qty
    sell_turnover = exit_price * qty
    turnover = buy_turnover + sell_turnover

    brokerage = min(20, buy_turnover * 0.0003) + min(20, sell_turnover * 0.0003)
    stt = sell_turnover * 0.00025
    txn = turnover * 0.0000345
    sebi = turnover * 0.000001
    stamp = buy_turnover * 0.00003
    gst = 0.18 * (brokerage + txn)

    return brokerage + stt + txn + sebi + stamp + gst

# =========================
# 🧠 TRADE ENGINE
# =========================
position = 0  # 1 long, -1 short
entry_price = 0

initial_capital = 100000
capital = initial_capital

positions = []
pnl_list = []

# Track trade details
entry_prices = []
exit_prices = []
trade_dirs = []  # 1 long, -1 short

for i in range(len(df)):
    
    row = df.loc[i]
    pnl = 0

    # ========= ENTRY =========
    if position == 0 and i > 0:
        
        if row['long_entry']:
            entry_price = row['close']
            position = 1

        elif row['short_entry']:
            entry_price = row['close']
            position = -1

    # ========= EXIT =========
    elif position != 0:
        
        prev_low = df.loc[i-1, 'low']
        prev_high = df.loc[i-1, 'high']
        exit_price = None

        # LONG EXIT
        if position == 1:
            if row['close'] < prev_low or row['close'] < row['prev_sma_5']:
                exit_price = row['close']
                gross_pnl = exit_price - entry_price

        # SHORT EXIT
        elif position == -1:
            if row['close'] > prev_high or row['close'] > row['prev_sma_5']:
                exit_price = row['close']
                gross_pnl = entry_price - exit_price

        if exit_price is not None:
            charges = calculate_charges(entry_price, exit_price, qty=1)
            net_pnl = gross_pnl - charges

            capital += net_pnl
            pnl = net_pnl

            # store trade
            entry_prices.append(entry_price)
            exit_prices.append(exit_price)
            trade_dirs.append(position)

            position = 0

    positions.append(position)
    pnl_list.append(pnl)

# =========================
# 💰 STORE RESULTS
# =========================
df['position'] = positions
df['pnl'] = pnl_list
df['equity'] = initial_capital + df['pnl'].cumsum()

# =========================
# 📊 TRADE DATAFRAME
# =========================
trades = pd.DataFrame({
    'entry': entry_prices,
    'exit': exit_prices,
    'dir': trade_dirs
})

# Difference (direction-aware)
trades['diff'] = np.where(
    trades['dir'] == 1,
    trades['exit'] - trades['entry'],
    trades['entry'] - trades['exit']
)

# =========================
# 📊 DIFF METRICS
# =========================
max_diff = trades['diff'].max() if len(trades) > 0 else 0
min_diff = trades['diff'].min() if len(trades) > 0 else 0
avg_diff = trades['diff'].mean() if len(trades) > 0 else 0

# =========================
# 📊 OTHER METRICS
# =========================
daily_pnl = df.groupby(df['date'].dt.date)['pnl'].sum()

rolling_max = df['equity'].cummax()
drawdown = df['equity'] / rolling_max - 1
max_drawdown = drawdown.min()

total_trades = len(trades)

# =========================
# 📦 FINAL REPORT
# =========================
final_capital = df['equity'].iloc[-1]

print("\n===== FINAL REPORT =====")
print(f"Final Capital: ₹{round(final_capital,2)}")
print(f"Return: {round((final_capital/initial_capital - 1)*100,2)}%")

print("\nMax Drawdown:", round(max_drawdown * 100, 2), "%")

print("\nTotal Trades:", total_trades)

print("\n===== PRICE DIFFERENCE METRICS =====")
print("Max Diff:", round(max_diff, 2))
print("Min Diff:", round(min_diff, 2))
print("Avg Diff:", round(avg_diff, 2))

print("\n===== DAILY PnL =====")
print(daily_pnl)


===== FINAL REPORT =====
Final Capital: ₹99457.22
Return: -0.54%

Max Drawdown: -0.55 %

Total Trades: 583

===== PRICE DIFFERENCE METRICS =====
Max Diff: 37.45
Min Diff: -43.75
Avg Diff: -0.19

===== DAILY PnL =====
date
2026-01-12     2.001597
2026-01-13    -1.038819
2026-01-14   -10.952414
2026-01-16    -0.317152
2026-01-19    -9.916676
                ...    
2026-04-13    -9.868119
2026-04-15    -2.277316
2026-04-16   -14.888755
2026-04-17    -6.727720
2026-04-20    -6.620448
Name: pnl, Length: 65, dtype: float64


In [10]:
import pandas as pd
import numpy as np

# =========================
# 🧹 PREP
# =========================
df = df.copy()
df = df.sort_values(by=['date', 'time']).reset_index(drop=True)

df['date'] = pd.to_datetime(df['date'])
df['datetime'] = pd.to_datetime(df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['time'])

# =========================
# 🎯 SIGNALS
# =========================
df['long_entry'] = (
    (df['close'] > df['prev_sma_5']) &
    (df['close'].shift(1) <= df['prev_sma_5'].shift(1))
)

df['short_entry'] = (
    (df['close'] < df['prev_sma_5']) &
    (df['close'].shift(1) >= df['prev_sma_5'].shift(1))
)

# =========================
# 💸 CHARGES (FYERS)
# =========================
def calculate_charges(entry_price, exit_price, qty=1):
    buy_turnover = entry_price * qty
    sell_turnover = exit_price * qty
    turnover = buy_turnover + sell_turnover

    brokerage = min(20, buy_turnover * 0.0003) + min(20, sell_turnover * 0.0003)
    stt = sell_turnover * 0.00025
    txn = turnover * 0.0000345
    sebi = turnover * 0.000001
    stamp = buy_turnover * 0.00003
    gst = 0.18 * (brokerage + txn)

    return brokerage + stt + txn + sebi + stamp + gst

# =========================
# 🧠 TRADE ENGINE
# =========================
position = 0
entry_price = 0

initial_capital = 100000
capital = initial_capital

positions = []
pnl_list = []

entry_prices = []
exit_prices = []
directions = []
trade_pnls = []

for i in range(len(df)):
    
    row = df.loc[i]
    pnl = 0

    # ===== ENTRY =====
    if position == 0 and i > 0:
        if row['long_entry']:
            entry_price = row['close']
            position = 1

        elif row['short_entry']:
            entry_price = row['close']
            position = -1

    # ===== EXIT =====
    elif position != 0:
        prev_low = df.loc[i-1, 'low']
        prev_high = df.loc[i-1, 'high']
        exit_price = None

        if position == 1:
            if row['close'] < prev_low or row['close'] < row['prev_sma_5']:
                exit_price = row['close']
                gross_pnl = exit_price - entry_price

        elif position == -1:
            if row['close'] > prev_high or row['close'] > row['prev_sma_5']:
                exit_price = row['close']
                gross_pnl = entry_price - exit_price

        if exit_price is not None:
            charges = calculate_charges(entry_price, exit_price, qty=1)
            net_pnl = gross_pnl - charges

            capital += net_pnl
            pnl = net_pnl

            # store trade
            entry_prices.append(entry_price)
            exit_prices.append(exit_price)
            directions.append(position)
            trade_pnls.append(net_pnl)

            position = 0

    positions.append(position)
    pnl_list.append(pnl)

# =========================
# 💰 RESULTS
# =========================
df['position'] = positions
df['pnl'] = pnl_list
df['equity'] = initial_capital + df['pnl'].cumsum()

# =========================
# 📊 TRADES DF
# =========================
trades = pd.DataFrame({
    'entry': entry_prices,
    'exit': exit_prices,
    'dir': directions,
    'pnl': trade_pnls
})

# Diff (pure price move)
trades['diff'] = np.where(
    trades['dir'] == 1,
    trades['exit'] - trades['entry'],
    trades['entry'] - trades['exit']
)

# =========================
# 📊 METRICS
# =========================
total_trades = len(trades)

win_trades = trades[trades['pnl'] > 0]
loss_trades = trades[trades['pnl'] < 0]

win_rate = (len(win_trades) / total_trades * 100) if total_trades > 0 else 0

avg_win = win_trades['pnl'].mean() if len(win_trades) > 0 else 0
avg_loss = loss_trades['pnl'].mean() if len(loss_trades) > 0 else 0

gross_profit = win_trades['pnl'].sum()
gross_loss = abs(loss_trades['pnl'].sum())

profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.nan

loss_rate = 1 - (win_rate / 100)
expectancy = (win_rate/100 * avg_win) - (loss_rate * abs(avg_loss))

trade_efficiency = trades['pnl'].mean() if total_trades > 0 else 0

# =========================
# 📉 DRAWDOWN
# =========================
rolling_max = df['equity'].cummax()
drawdown = df['equity'] / rolling_max - 1
max_drawdown = drawdown.min()

# =========================
# 📊 SHARPE
# =========================
returns = df['pnl']

if returns.std() != 0:
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252 * 375)
else:
    sharpe = 0

# =========================
# 📊 DAILY METRICS
# =========================
daily_pnl = df.groupby(df['date'].dt.date)['pnl'].sum()
trades_per_day = trades.groupby(df.loc[df['pnl'] != 0, 'date'].dt.date).size()

# =========================
# 📊 DIFF METRICS
# =========================
max_diff = trades['diff'].max() if total_trades > 0 else 0
min_diff = trades['diff'].min() if total_trades > 0 else 0
avg_diff = trades['diff'].mean() if total_trades > 0 else 0

# =========================
# 📦 FINAL REPORT
# =========================
final_capital = df['equity'].iloc[-1]
total_profit = final_capital - initial_capital
return_pct = (total_profit / initial_capital) * 100

print("\n===== FINAL REPORT =====")
print(f"Initial Capital: ₹{initial_capital}")
print(f"Final Capital: ₹{round(final_capital,2)}")
print(f"Total Profit: ₹{round(total_profit,2)}")
print(f"Return: {round(return_pct,2)}%")

print("\n===== RISK =====")
print("Max Drawdown:", round(max_drawdown * 100, 2), "%")
print("Sharpe Ratio:", round(sharpe, 2))

print("\n===== TRADE METRICS =====")
print("Total Trades:", total_trades)
print("Win Rate:", round(win_rate, 2), "%")
print("Avg Win (₹):", round(avg_win, 2))
print("Avg Loss (₹):", round(avg_loss, 2))
print("Profit Factor:", round(profit_factor, 2))
print("Expectancy (₹):", round(expectancy, 2))
print("Trade Efficiency (₹):", round(trade_efficiency, 2))

print("\n===== PRICE DIFFERENCE METRICS =====")
print("Max Diff:", round(max_diff, 2))
print("Min Diff:", round(min_diff, 2))
print("Avg Diff:", round(avg_diff, 2))

print("\n===== TRADES PER DAY =====")
print(trades_per_day)

print("\n===== DAILY PnL =====")
print(daily_pnl)


===== FINAL REPORT =====
Initial Capital: ₹100000
Final Capital: ₹99457.22
Total Profit: ₹-542.78
Return: -0.54%

===== RISK =====
Max Drawdown: -0.55 %
Sharpe Ratio: -22.3

===== TRADE METRICS =====
Total Trades: 583
Win Rate: 19.38 %
Avg Win (₹): 4.22
Avg Loss (₹): -2.17
Profit Factor: 0.47
Expectancy (₹): -0.93
Trade Efficiency (₹): -0.93

===== PRICE DIFFERENCE METRICS =====
Max Diff: 37.45
Min Diff: -43.75
Avg Diff: -0.19

===== TRADES PER DAY =====
date
2026-01-12     8
2026-01-13     6
2026-01-14     8
2026-01-16     6
2026-01-19     9
2026-01-20    10
2026-01-21     6
2026-01-22    11
2026-01-23     4
dtype: int64

===== DAILY PnL =====
date
2026-01-12     2.001597
2026-01-13    -1.038819
2026-01-14   -10.952414
2026-01-16    -0.317152
2026-01-19    -9.916676
                ...    
2026-04-13    -9.868119
2026-04-15    -2.277316
2026-04-16   -14.888755
2026-04-17    -6.727720
2026-04-20    -6.620448
Name: pnl, Length: 65, dtype: float64
